# F5-probability — Practice p21

**Type:** constrained coding · **Difficulty:** core · **Concepts:** conditional-probability, bayes-rule

Implement:

1. `conditional_rate(event, given)`: coerce inputs to Boolean arrays, require same-shape one-dimensional masks, return empirical $P(\text{event}\mid\text{given})$, and raise `ValueError` for shape violations or zero conditioning count.
2. `bayes_binary(p_e_given_h, p_h, p_e_given_not_h)`: return $P(H\mid E)$ using total probability. Raise `ValueError` unless all rates lie in $[0,1]$ and the evidence denominator is positive.

Then compute `empirical_p_rain_given_alert` and `bayes_p_rain_given_alert`.

**Zero points:** loops/comprehensions; `np.mean`/`np.average`; pandas; division by the full sample size. Bayes direction matters: return $P(H\mid E)$, not $P(E\mid H)$.


In [ ]:
import numpy as np

rain = np.array([1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0], dtype=bool)
alert = np.array([1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0], dtype=bool)

def conditional_rate(event, given):
    event_mask = np.asarray(event, dtype=bool)
    given_mask = np.asarray(given, dtype=bool)
    if event_mask.ndim != 1 or given_mask.ndim != 1 or event_mask.shape != given_mask.shape:
        raise ValueError("event and given must be same-shape one-dimensional masks")
    given_count = int(np.count_nonzero(given_mask))
    if given_count == 0:
        raise ValueError("conditioning count must be positive")
    joint_count = int(np.count_nonzero(event_mask & given_mask))
    return joint_count / given_count

def bayes_binary(p_e_given_h, p_h, p_e_given_not_h):
    if not (0.0 <= p_e_given_h <= 1.0 and
            0.0 <= p_h <= 1.0 and
            0.0 <= p_e_given_not_h <= 1.0):
        raise ValueError("all rates must lie in [0, 1]")
    evidence = p_e_given_h * p_h + p_e_given_not_h * (1.0 - p_h)
    if evidence <= 0.0:
        raise ValueError("evidence probability must be positive")
    return p_e_given_h * p_h / evidence

empirical_p_rain_given_alert = conditional_rate(rain, alert)
bayes_p_rain_given_alert = bayes_binary(0.80, 0.25, 0.10)

In [ ]:
# Immutable semantic checks: recompute from the stated arrays and Bayes inputs.
_expected_empirical = int((rain & alert).sum()) / int(alert.sum())
_p_e_given_h = 0.80
_p_h = 0.25
_p_e_given_not_h = 0.10
_expected_evidence = _p_e_given_h * _p_h + _p_e_given_not_h * (1.0 - _p_h)
_expected_bayes = _p_e_given_h * _p_h / _expected_evidence
_expected_reverse = int((alert & rain).sum()) / int(rain.sum())
assert np.isclose(empirical_p_rain_given_alert, _expected_empirical, atol=1e-12, rtol=0.0)
assert np.isclose(bayes_p_rain_given_alert, _expected_bayes, atol=1e-12, rtol=0.0)
assert np.isclose(conditional_rate(alert, rain), _expected_reverse, atol=1e-12, rtol=0.0)
# Valid Python list inputs must be coerced before shape/ndim inspection.
_list_event = [True, False, True, True]
_list_given = [True, True, False, True]
_list_event_bool = np.asarray(_list_event, dtype=bool)
_list_given_bool = np.asarray(_list_given, dtype=bool)
_expected_list_rate = np.count_nonzero(_list_event_bool & _list_given_bool) / np.count_nonzero(_list_given_bool)
assert np.isclose(conditional_rate(_list_event, _list_given), _expected_list_rate, atol=1e-12, rtol=0.0)
# Non-Boolean numeric arrays must use ordinary truth-value coercion.
_numeric_event = np.array([2.0, 0.0, 0.0, 0.0, 5.0])
_numeric_given = np.array([0, 4, -1, 0, 2])
_numeric_event_bool = np.asarray(_numeric_event, dtype=bool)
_numeric_given_bool = np.asarray(_numeric_given, dtype=bool)
_expected_numeric_rate = np.count_nonzero(_numeric_event_bool & _numeric_given_bool) / np.count_nonzero(_numeric_given_bool)
assert np.isclose(conditional_rate(_numeric_event, _numeric_given), _expected_numeric_rate, atol=1e-12, rtol=0.0)
try:
    conditional_rate(rain[:-1], alert)
except ValueError:
    pass
else:
    raise AssertionError("mismatched shapes must raise ValueError")
try:
    conditional_rate(rain.reshape(3, 4), alert.reshape(3, 4))
except ValueError:
    pass
else:
    raise AssertionError("2-D inputs must raise ValueError")
try:
    conditional_rate(rain, np.zeros_like(rain))
except ValueError:
    pass
else:
    raise AssertionError("zero-probability conditioning must raise ValueError")
try:
    bayes_binary(-0.1, 0.2, 0.3)
except ValueError:
    pass
else:
    raise AssertionError("rates below 0 must raise ValueError")
try:
    bayes_binary(0.8, 1.1, 0.2)
except ValueError:
    pass
else:
    raise AssertionError("rates above 1 must raise ValueError")
try:
    bayes_binary(0.0, 0.5, 0.0)
except ValueError:
    pass
else:
    raise AssertionError("zero evidence probability must raise ValueError")
print("p21 checks passed")


The empirical calculation divides the four rain-and-alert days by the six alert days, giving $P(\text{rain}\mid\text{alert})=2/3$; reversing the direction instead divides by the five rain days and gives $4/5$. For Bayes' rule, the numerator is $P(E\mid H)P(H)=0.80(0.25)$, while the evidence denominator includes both base-rate routes, $0.80(0.25)+0.10(0.75)=0.275$, so $P(H\mid E)=8/11$. Sensitivity $P(E\mid H)$ is therefore not the posterior $P(H\mid E)$: the prior and false-positive route are essential.

### Answer check

The immutable assertions recompute both conditional directions and the Bayes posterior with `atol=1e-12`, `rtol=0.0`; they also verify vectorized Boolean coercion for Python lists and non-Boolean numeric arrays, mismatched shapes, two-dimensional inputs, zero conditioning mass, out-of-range rates, and zero evidence. The implementation uses vectorized counts only—no loops, comprehensions, full-sample denominator, or banned averaging API.